**Purpose:** FINAL LSTM allocating via predicted returns (technical only).

**Outputs:** `LSTMvia-returns_technical_weights.npy`, `summary_metrics.csv`, `test_bl_returns.csv`, `test_bl_weights.csv`, `{prefix}_cumulative_returns.png`

**Notes:** paths resolve via `src.config` (run `pip install -e .` from the repo root first).


In [ ]:
from src.seeds import SEED_PORTFOLIO_LSTM_RETURNS_V5_2_2, SEED_PORTFOLIO_LSTM_RETURNS_V5_2_2_BL


In [ ]:
from src.config import PROJECT_ROOT


# LSTM + Black–Litterman Portfolio — Returns Prediction

Multi-output next-day returns prediction with a shared per-asset LSTM encoder,
followed by Black–Litterman portfolio construction using the ensemble of top-N models.

**Pipeline**
1. Feature engineering (per-asset + market, per-bucket Optuna selection)
2. Stationary transforms fit on training window only (robust z-score / log1p)
3. Shared LSTM encoder + cross-asset Transformer mixer → per-asset return forecasts
4. Optuna HPO over 3 walk-forward CV folds (maximises mean IC Spearman)
5. Refit top-N trials → ensemble predictions
6. Black–Litterman posterior → mean-variance weights
7. BL hyperparameter search on validation Sharpe
8. Final test with mid-test refit (2023–2024, 2024–2025)

**Walk-forward CV folds** (expanding window, no data leakage)
| Fold | Train | Val |
|------|-------|-----|
| 1 | 2015-07 → 2020-06 | 2020-07 → 2021-06 |
| 2 | 2015-07 → 2021-06 | 2021-07 → 2022-06 |
| 3 | 2015-07 → 2022-06 | 2022-07 → 2023-06 |

Test is strictly out-of-sample: **2023-07 → 2025-06** (2-year, mid-test refit).


- In forecasting the prices of multiple sectoral ETFs, a single LSTM model is an optimal approach due to its ability to leverage shared economic factors influencing all sectors, while maintaining flexibility to adapt to sector-specific behaviors. Sector ETFs are often correlated through common macroeconomic drivers, such as interest rates or commodity prices, and using a unified model enables the sharing of learned representations across sectors, improving generalization and reducing overfitting, especially in cases where data for some sectors may be limited. By incorporating sector embeddings as an input feature, the LSTM model can capture unique sectoral dynamics while still benefiting from the broader, cross-sector knowledge. This approach aligns with multi-task learning principles, where a single model can simultaneously optimize for multiple tasks (sector predictions), balancing efficiency, scalability, and accuracy. Empirical evidence from financial forecasting supports the efficacy of such global models, which often outperform separate models due to their ability to generalize across correlated assets. Thus, a single LSTM model with sector embeddings is both a computationally efficient and statistically robust solution for multi-sector ETF price prediction.

- Hyperparameters were selected using an expanding-window walk-forward validation scheme. Three folds were constructed: training on 5, 6, and 7 years of historical data respectively, each followed by a 1-year validation period. The final model was trained on the full 8-year pre-test period and evaluated on a strictly out-of-sample 2-year test set (2023–2025). This preserves temporal order and avoids information leakage common in random cross-validation for time-series data (Hyndman & Athanasopoulos). Some sources: [
0df79,
[00001](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html?utm_source=chatgpt.com),
[00002](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=3104847),
4061c
]

- FALTA:

    - falar das features e transformacoes

    - cross validation ja falei mas confirmar se estar ok

    - tem early stopping? rever codigo

    - que metricas foram usadas para otimizacao e isso? n usar so rmse, IC é fixe pq da ideia de direcao e como os assets se comportam relativamente uns aos outros, mas rmse é importante para avaliar a qualidade da previsao em termos de valor absoluto

    - ideia é usar top N modelos para portfolio opimzation (media = preço, variancia = incerteza la para o black litterman) -> acabar por usar mais features, vantagem lstm vs drl, mas drl é mt mais complexo (O complexity bla bla) ent em termos de tempo é oq é é lidar....

    - missing featudares, como lida?

    - what is the loss

    - ns se gosto da loss nem do early stop definido

    - aos 20 trails ja bateu quase 1 sharpe

    - N=5? pq nao 3 ou 10

    - 83780? 4061c? 2b4c0?

## 1) Imports

In [ ]:
# lstm_returns_prediction_v3.py
# ------------------------------------------------------------
# Multi-output next-day returns prediction with:
# - Walk-forward splits
# - Optuna Bayesian HPO + feature selection
# - Per-feature transforms fit on TRAIN only
# - Shared per-asset temporal encoder + cross-asset fusion head
# - Masked loss for missing targets
# - Cost-aware training target via per-asset spreads
# - Raw realized returns preserved for covariance / BL backtest / reporting
# - Minimal diagnostics only:
#     * cumulative returns plot (all assets in one chart)
#     * per-asset metrics JSON
# - Extra finite checks to avoid NaN training
# ------------------------------------------------------------

from __future__ import annotations

import os
import json
import copy
import random
import logging
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd

import optuna
from optuna.samplers import TPESampler
from optuna.trial import TrialState

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

## 2) Config

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────
DATASET_PATH = str(PROJECT_ROOT / "03_portfolio/dataset.parquet")
SPREADS_PATH = str(PROJECT_ROOT / "01_data/aux/bid-ask_spread.json")

# ── Universe ─────────────────────────────────────────────────────────────
ASSETS = ['XLB', 'XLC', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLRE', 'XLU', 'XLV', 'XLY']

# ── Feature buckets (Optuna picks one per bucket each trial) ─────────────
FEATURES = {
    'Price':      ['Close'],
    'OHLC':       ['Open', 'High', 'Low'],
    'Trend':      ['SMA5', 'SMA22', 'SMA63', 'SMA126', 'SMA252', 'MACD', 'ADX'],
    'Momentum':   ['RSI', 'MACD', 'CCI'],
    'Volume':     ['Volume', 'OBV', 'ADI', 'VolumeVariation'],
    'Volatility': ['BBUpper', 'BBLower', '5dayVolatility', '22dayVolatility',
                   '63dayVolatility', 'RatioVolatility'],
    'Market':     ['VIXIndexClose'],
}

# ── Study settings ────────────────────────────────────────────────────────
OUT_DIR       = 'v522'
N_TRIALS      = 0          # see estimate_search_space() below for guidance
N_TRIALS_BL   = 0          # BL HPO trials
MODEL_SEEDS   = [11, 22, 33]
BL_SEED       = SEED_PORTFOLIO_LSTM_RETURNS_V5_2_2_BL


## 3) Utilities

In [ ]:

def setup_logger(log_dir: str) -> logging.Logger:
    #os.makedirs(log_dir, exist_ok=True)
    logger = logging.getLogger(f"joint_asset_lstm_{os.path.basename(log_dir)}")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    fmt = logging.Formatter("[%(asctime)s] [%(levelname)s] %(message)s")

    ch = logging.StreamHandler()
    ch.setFormatter(fmt)
    logger.addHandler(ch)

    #fh = logging.FileHandler(os.path.join(log_dir, "run.log"))
    #fh.setFormatter(fmt)
    #logger.addHandler(fh)

    logger.propagate = False
    return logger

def seed_everything(seed: int):
    """Global seed for reproducibility across Python, NumPy, and PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def safe_makedirs(path: str):
    os.makedirs(path, exist_ok=True)


## 4) Walk-forward splits

In [ ]:
# Expanding-window folds: each fold adds one year to the training window.
# Transforms are always fit on the training portion only (no lookahead leakage).

def make_walk_forward_splits(dates_index: pd.DatetimeIndex):
    splits = []
    periods = [
        (pd.Timestamp("2015-07-01"), pd.Timestamp("2020-06-30"),
         pd.Timestamp("2020-07-01"), pd.Timestamp("2021-06-30")),
        (pd.Timestamp("2015-07-01"), pd.Timestamp("2021-06-30"),
         pd.Timestamp("2021-07-01"), pd.Timestamp("2022-06-30")),
        (pd.Timestamp("2015-07-01"), pd.Timestamp("2022-06-30"),
         pd.Timestamp("2022-07-01"), pd.Timestamp("2023-06-30")),
    ]

    for tr_start, tr_end, v_start, v_end in periods:
        train_idx = np.where((dates_index >= tr_start) & (dates_index <= tr_end))[0]
        val_idx = np.where((dates_index >= v_start) & (dates_index <= v_end))[0]

        if len(train_idx) == 0 or len(val_idx) == 0:
            print(
                f"[SKIP SPLIT] train {tr_start.date()}-{tr_end.date()} "
                f"val {v_start.date()}-{v_end.date()} | "
                f"lens train={len(train_idx)} val={len(val_idx)}"
            )
            continue

        splits.append((train_idx, val_idx))

    if len(splits) == 0:
        raise ValueError("No valid walk-forward splits. Dataset date coverage doesn't match the hard-coded periods.")

    return splits


def split_train_for_early_stop(
    train_idx: np.ndarray,
    dates_index: pd.DatetimeIndex,
    es_years: int = 1,
    min_train_days: int = 252,
) -> Tuple[np.ndarray, np.ndarray]:
    train_dates = dates_index[train_idx]
    train_end = train_dates[-1]
    es_start_date = train_end - pd.DateOffset(years=es_years) + pd.Timedelta(days=1)

    es_mask = train_dates >= es_start_date
    core_mask = ~es_mask

    core_idx = train_idx[core_mask]
    es_idx = train_idx[es_mask]

    if len(core_idx) < min_train_days or len(es_idx) < min_train_days // 2:
        cut = max(min_train_days, len(train_idx) - 252 * es_years)
        cut = min(cut, len(train_idx) - 1)
        core_idx = train_idx[:cut]
        es_idx = train_idx[cut:]

    if len(core_idx) == 0 or len(es_idx) == 0:
        raise ValueError("Failed to create non-empty early-stopping split inside training window.")

    return core_idx, es_idx

## 5) Feature engineering

In [ ]:

def pick_features_from_buckets(trial: optuna.Trial, features: Dict[str, List[str]]) -> Dict[str, Any]:
    """
    - Price: always use log_ret_1d
    - OHLC: on/off as group
    - Trend/Momentum/Volume: pick one each
    - Volatility: allow BBands as one choice -> [BBLower, BBUpper]
    - Market: on/off
    """
    picks = {}
    picks["Price"] = "log_ret_1d"

    use_ohlc = trial.suggest_categorical("use_ohlc", [0, 1])
    picks["OHLC"] = ["Open", "High", "Low"] if use_ohlc == 1 else []

    for bucket in ["Trend", "Momentum", "Volume"]:
        picks[bucket] = trial.suggest_categorical(f"pick_{bucket}", features[bucket])

    vol_options = [f for f in features["Volatility"] if f not in ("BBLower", "BBUpper")]
    vol_options.append("BBands")

    vol_choice = trial.suggest_categorical("pick_Volatility", vol_options)
    if vol_choice == "BBands":
        picks["Volatility"] = ["BBLower", "BBUpper"]
    else:
        picks["Volatility"] = vol_choice

    use_market = trial.suggest_categorical("use_market", [0, 1])
    picks["Market"] = "VIXIndexClose" if use_market == 1 else None
    return picks


def get_per_asset_feature_list(picks: Dict[str, Any]) -> List[str]:
    feats = ["log_ret_1d"]

    for f in picks.get("OHLC", []):
        feats.append(f)

    for bucket in ["Trend", "Momentum", "Volume", "Volatility"]:
        val = picks[bucket]
        if isinstance(val, list):
            feats.extend(val)
        else:
            feats.append(val)

    out = []
    seen = set()
    for f in feats:
        if f not in seen:
            out.append(f)
            seen.add(f)
    return out


def required_df_columns(assets: List[str], per_asset_feats: List[str], picks: Dict[str, Any]) -> List[str]:
    cols = []
    for a in assets:
        cols.append(f"{a}_Close")
        for f in per_asset_feats:
            if f == "log_ret_1d":
                continue
            cols.append(f"{a}_{f}")
    if picks.get("Market", None) is not None:
        cols.append(picks["Market"])
    return cols

## 6) Stationary transforms

In [ ]:
# Each feature gets a transform spec assigned by name, then fit on the training window.
# All clipping and centering is derived from training data only.
#
# Transform kinds:
#   robust_z        — winsorise → (x - median) / (1.4826 × MAD)
#   log1p_robust_z  — same but apply sign(x)·log1p(|x|) first (for volume-like data)
#   bounded_0_100   — (x - 50) / 50  (for RSI, ADX which live in [0, 100])

def _winsorize_fit(x: np.ndarray, q: float) -> Tuple[float, float]:
    lo = np.nanquantile(x, q)
    hi = np.nanquantile(x, 1.0 - q)
    if not np.isfinite(lo):
        lo = 0.0
    if not np.isfinite(hi):
        hi = 0.0
    if hi < lo:
        hi = lo
    return float(lo), float(hi)


def _winsorize_apply(x: np.ndarray, lo: float, hi: float) -> np.ndarray:
    return np.clip(x, lo, hi)


@dataclass
class FeatureTransformSpec:
    kind: str
    winsor_q: float = 0.01
    clip_value: float = 8.0


@dataclass
class FittedTransform:
    spec: FeatureTransformSpec
    lo: float = 0.0
    hi: float = 0.0
    center: float = 0.0
    scale: float = 1.0

    def transform(self, x: np.ndarray) -> np.ndarray:
        x = x.astype(np.float32)
        x = np.nan_to_num(x, nan=np.nan, posinf=np.nan, neginf=np.nan)

        if self.spec.kind in ("robust_z", "log1p_robust_z"):
            x = _winsorize_apply(x, self.lo, self.hi)

        if self.spec.kind == "identity":
            y = x
        elif self.spec.kind == "bounded_0_100":
            y = (x - 50.0) / 50.0
        elif self.spec.kind == "log1p_robust_z":
            y = np.sign(x) * np.log1p(np.abs(x))
            y = (y - self.center) / self.scale
        elif self.spec.kind == "robust_z":
            y = (x - self.center) / self.scale
        else:
            raise ValueError(f"Unknown transform kind: {self.spec.kind}")

        y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
        y = np.clip(y, -self.spec.clip_value, self.spec.clip_value)
        return y.astype(np.float32)


def default_transform_spec(feature_name: str) -> FeatureTransformSpec:
    name = feature_name.lower()

    if feature_name == "log_ret_1d":
        return FeatureTransformSpec(kind="robust_z", winsor_q=0.01, clip_value=8.0)

    if name in ("rsi", "adx"):
        return FeatureTransformSpec(kind="bounded_0_100", clip_value=2.0)

    if name in ("volume", "volumevariation", "obv", "adi"):
        return FeatureTransformSpec(kind="log1p_robust_z", winsor_q=0.01, clip_value=8.0)

    if name in ("open", "high", "low"):
        return FeatureTransformSpec(kind="robust_z", winsor_q=0.01, clip_value=8.0)

    return FeatureTransformSpec(kind="robust_z", winsor_q=0.01, clip_value=8.0)


def fit_transforms_on_train(
    X_train_raw: np.ndarray,              # [T,N,F]
    feat_names: List[str],
    market_train_raw: Optional[np.ndarray],
) -> Tuple[List[FittedTransform], Optional[FittedTransform]]:
    fitted = []

    for j, fname in enumerate(feat_names):
        spec = default_transform_spec(fname)
        vals = X_train_raw[:, :, j].reshape(-1)
        vals = vals[np.isfinite(vals)]

        if vals.size == 0:
            fitted.append(FittedTransform(spec=spec))
            continue

        if spec.kind in ("robust_z", "log1p_robust_z"):
            lo, hi = _winsorize_fit(vals, spec.winsor_q)
            vals_w = _winsorize_apply(vals, lo, hi)
            if spec.kind == "log1p_robust_z":
                vals_w = np.sign(vals_w) * np.log1p(np.abs(vals_w))

            center = float(np.nanmedian(vals_w))
            mad = float(np.nanmedian(np.abs(vals_w - center)))
            scale = 1.4826 * mad
            if not np.isfinite(scale) or scale < 1e-6:
                scale = float(np.nanstd(vals_w))
            if not np.isfinite(scale) or scale < 1e-6:
                scale = 1.0

            fitted.append(FittedTransform(
                spec=spec, lo=lo, hi=hi, center=center, scale=scale
            ))

        elif spec.kind in ("bounded_0_100", "identity"):
            fitted.append(FittedTransform(spec=spec))
        else:
            raise ValueError(spec.kind)

    market_ft = None
    if market_train_raw is not None:
        spec = FeatureTransformSpec(kind="robust_z", winsor_q=0.01, clip_value=8.0)
        v = market_train_raw[:, 0]
        v = v[np.isfinite(v)]
        if v.size > 2:
            lr = np.log(v[1:] / v[:-1])
            lr = lr[np.isfinite(lr)]
        else:
            lr = np.array([0.0], dtype=np.float32)

        lo, hi = _winsorize_fit(lr, spec.winsor_q)
        lr_w = _winsorize_apply(lr, lo, hi)
        center = float(np.nanmedian(lr_w))
        mad = float(np.nanmedian(np.abs(lr_w - center)))
        scale = 1.4826 * mad
        if not np.isfinite(scale) or scale < 1e-6:
            scale = float(np.nanstd(lr_w))
        if not np.isfinite(scale) or scale < 1e-6:
            scale = 1.0

        market_ft = FittedTransform(spec=spec, lo=lo, hi=hi, center=center, scale=scale)

    return fitted, market_ft


def apply_transforms(
    X_raw: np.ndarray,
    fitted: List[FittedTransform],
    market_raw: Optional[np.ndarray],
    market_ft: Optional[FittedTransform],
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    X = np.empty_like(X_raw, dtype=np.float32)
    for j, ft in enumerate(fitted):
        X[:, :, j] = ft.transform(X_raw[:, :, j])

    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    m_out = None
    if market_raw is not None and market_ft is not None:
        v = market_raw[:, 0].astype(np.float32)
        lr = np.full_like(v, np.nan, dtype=np.float32)
        valid = np.isfinite(v[1:]) & np.isfinite(v[:-1]) & (v[1:] > 0) & (v[:-1] > 0)
        lr[1:][valid] = np.log(v[1:][valid] / v[:-1][valid])
        lr = np.nan_to_num(lr, nan=0.0, posinf=0.0, neginf=0.0)
        m_out = market_ft.transform(lr).reshape(-1, 1).astype(np.float32)
        m_out = np.nan_to_num(m_out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    return X, m_out

## 7) Raw arrays & targets

In [ ]:
# build_raw_arrays returns two target arrays:
#   y_model  — spread-adjusted return used as the MSE/Huber training target
#   y_raw    — raw close-to-close return used for BL covariance and backtest metrics

def apply_spread_to_target(raw_ret: np.ndarray, spread: float) -> np.ndarray:
    return np.sign(raw_ret) * np.maximum(np.abs(raw_ret) - spread, 0.0)


def build_raw_arrays(
    df: pd.DataFrame,
    assets: List[str],
    picks: Dict[str, Any],
    features: Dict[str, List[str]],
    logger: logging.Logger,
    spreads: Optional[np.ndarray] = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, List[str], Optional[np.ndarray]]:
    """
    Returns:
      closes_raw: [T,N]
      X_raw:      [T,N,F]
      y_model:    [T,N] spread-adjusted training target
      y_raw:      [T,N] actual next-day close-to-close return from t -> t+1
      y_mask:     [T,N] valid target mask
      feat_names
      market_raw: [T,1] or None
    """
    per_asset_feats = get_per_asset_feature_list(picks)
    cols = required_df_columns(assets, per_asset_feats, picks)

    if spreads is None:
        spreads = np.zeros(len(assets), dtype=np.float32)
    else:
        spreads = np.asarray(spreads, dtype=np.float32)
        if spreads.shape[0] != len(assets):
            raise ValueError(f"spreads length {len(spreads)} != number of assets {len(assets)}")

    missing = [c for c in cols if c not in df.columns]
    if missing:
        logger.error(f"Missing columns ({len(missing)}): {missing[:25]}{'...' if len(missing) > 25 else ''}")
        raise KeyError("Dataset missing required columns for this feature set.")

    T = len(df)
    N = len(assets)
    F = len(per_asset_feats)

    closes = np.zeros((T, N), dtype=np.float32)
    X_raw = np.zeros((T, N, F), dtype=np.float32)

    for i, a in enumerate(assets):
        c = df[f"{a}_Close"].to_numpy(np.float32)
        closes[:, i] = c

        for j, f in enumerate(per_asset_feats):
            if f == "log_ret_1d":
                lr = np.full_like(c, np.nan, dtype=np.float32)
                valid = (c[1:] > 0) & (c[:-1] > 0) & np.isfinite(c[1:]) & np.isfinite(c[:-1])
                lr[1:][valid] = np.log(c[1:][valid] / c[:-1][valid])
                X_raw[:, i, j] = lr

            elif f in ("Open", "High", "Low"):
                raw = df[f"{a}_{f}"].to_numpy(np.float32)
                rel = np.full_like(raw, np.nan, dtype=np.float32)
                valid = (raw > 0) & (c > 0) & np.isfinite(raw) & np.isfinite(c)
                rel[valid] = np.log(raw[valid] / c[valid])
                X_raw[:, i, j] = rel

            elif f.startswith("SMA"):
                sma = df[f"{a}_{f}"].to_numpy(np.float32)
                rel = np.full_like(sma, np.nan, dtype=np.float32)
                valid = (sma > 0) & (c > 0) & np.isfinite(sma) & np.isfinite(c)
                rel[valid] = np.log(c[valid] / sma[valid])
                X_raw[:, i, j] = rel

            else:
                X_raw[:, i, j] = df[f"{a}_{f}"].to_numpy(np.float32)

    y_model = np.full((T, N), np.nan, dtype=np.float32)
    y_raw = np.full((T, N), np.nan, dtype=np.float32)
    valid_y = np.zeros((T, N), dtype=bool)

    for i in range(N):
        c = closes[:, i]
        valid = (
            np.isfinite(c[:-1]) & np.isfinite(c[1:]) &
            (c[:-1] > 0.0) & (c[1:] > 0.0)
        )

        tmp_raw = np.full(T - 1, np.nan, dtype=np.float32)
        tmp_model = np.full(T - 1, np.nan, dtype=np.float32)

        raw_ret = (c[1:][valid] / c[:-1][valid]) - 1.0
        spread_i = float(spreads[i])
        model_ret = apply_spread_to_target(raw_ret, spread_i)

        tmp_raw[valid] = raw_ret.astype(np.float32)
        tmp_model[valid] = model_ret.astype(np.float32)

        y_raw[:-1, i] = tmp_raw
        y_model[:-1, i] = tmp_model
        valid_y[:-1, i] = valid

    y_raw = np.where(np.isfinite(y_raw), y_raw, np.nan).astype(np.float32)
    y_model = np.where(np.isfinite(y_model), y_model, np.nan).astype(np.float32)

    market = None
    if picks.get("Market", None) is not None:
        market = df[picks["Market"]].to_numpy(np.float32).reshape(-1, 1)

    return closes, X_raw, y_model, y_raw, valid_y, per_asset_feats, market


def compute_sample_validity(
    X_raw: np.ndarray,      # [T,N,F]
    y_mask: np.ndarray,     # [T,N]
    lookback: int,
    min_valid_targets: int = 1,
) -> np.ndarray:
    """
    A sample at time t is valid if:
    - lookback window fully available for all selected per-asset features across all assets
    - and at least min_valid_targets assets have valid next-day returns
    """
    T, N, F = X_raw.shape
    finite_feats = np.isfinite(X_raw).all(axis=2)  # [T,N]
    sample_ok = np.zeros(T, dtype=bool)

    for t in range(lookback - 1, T - 1):
        win_ok = finite_feats[t - lookback + 1: t + 1].all()
        target_ok = int(y_mask[t].sum()) >= min_valid_targets
        sample_ok[t] = bool(win_ok and target_ok)

    return sample_ok

## 8) Dataset & DataLoader

In [ ]:
class SequenceDataset(Dataset):
    """
    Each sample:
        x      : [L, N, F]  feature window
        y      : [N]        training target (y_model, spread-adjusted)
        mask   : [N]        1 where target is valid (ETF was trading)
        t      : int        absolute time index
    """
class SequenceDataset(Dataset):
    def __init__(
        self,
        X: np.ndarray,                # [T,N,F]
        y: np.ndarray,                # [T,N]
        y_mask: np.ndarray,           # [T,N]
        valid_t: np.ndarray,
        lookback: int,
        market: Optional[np.ndarray] = None,   # [T,1]
    ):
        self.X = X
        self.y = y
        self.y_mask = y_mask
        self.valid_t = np.asarray(valid_t, dtype=np.int64)
        self.lookback = int(lookback)
        self.market = market

    def __len__(self):
        return len(self.valid_t)

    def __getitem__(self, idx):
        t = int(self.valid_t[idx])
        x = self.X[t - self.lookback + 1: t + 1]  # [L,N,F]

        if self.market is not None:
            m = self.market[t - self.lookback + 1: t + 1]       # [L,1]
            m = np.repeat(m[:, None, :], x.shape[1], axis=1)    # [L,N,1]
            x = np.concatenate([x, m], axis=2)

        y = self.y[t].copy()          # [N]
        m = self.y_mask[t].copy()     # [N]

        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
        m = np.nan_to_num(m, nan=0.0, posinf=0.0, neginf=0.0)

        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32),
            torch.tensor(m, dtype=torch.float32),
            torch.tensor(t, dtype=torch.long),
        )



def make_loader(
    X: np.ndarray,
    y: np.ndarray,
    y_mask: np.ndarray,
    valid_t: np.ndarray,
    lookback: int,
    batch_size: int,
    market: Optional[np.ndarray],
    shuffle: bool,
) -> DataLoader:
    ds = SequenceDataset(
        X=X,
        y=y,
        y_mask=y_mask,
        valid_t=valid_t,
        lookback=lookback,
        market=market,
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=False)

## 9) Model — JointAssetLSTM

In [ ]:
# Architecture:
#   1. Shared per-asset LSTM encoder (weights shared across all assets)
#   2. Optional linear projection per asset
#   3. CrossAssetMixer (Transformer) for cross-sectional information sharing
#   4. Joint MLP fusion head → per-asset return forecasts
#   5. Optional residual connection from per-asset embeddings

class CrossAssetMixer(nn.Module):
    """
    Lightweight cross-asset mixer over asset embeddings [B, N, D].
    Uses TransformerEncoder across asset dimension.
    """
    def __init__(self, d_model: int, num_heads: int, dropout: float, num_layers: int):
        super().__init__()
        if num_layers <= 0:
            self.net = nn.Identity()
        else:
            enc_layer = nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=num_heads,
                dim_feedforward=max(4 * d_model, 64),
                dropout=dropout,
                batch_first=True,
                activation="gelu",
                norm_first=True,
            )
            self.net = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class JointAssetLSTM(nn.Module):
    """
    Shared per-asset temporal encoder + optional cross-asset attention + joint fusion head.

    Input:
        x: [B, L, N, F]
    Output:
        yhat: [B, N]
    """
    def __init__(
        self,
        input_dim_per_asset: int,
        num_assets: int,
        hidden_size: int,
        num_layers: int,
        dropout: float,
        bidirectional: bool = False,
        encoder_proj: int = 0,
        asset_mixer_layers: int = 1,
        asset_mixer_heads: int = 4,
        fusion_hidden: int = 64,
        use_residual_fusion: bool = True,
    ):
        super().__init__()

        self.num_assets = num_assets

        lstm_dropout = dropout if num_layers > 1 else 0.0
        self.encoder = nn.LSTM(
            input_size=input_dim_per_asset,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=lstm_dropout,
            batch_first=True,
            bidirectional=bidirectional,
        )

        enc_dim = hidden_size * (2 if bidirectional else 1)

        if encoder_proj > 0:
            self.asset_proj = nn.Sequential(
                nn.Linear(enc_dim, encoder_proj),
                nn.GELU(),
                nn.Dropout(dropout),
            )
            asset_dim = encoder_proj
        else:
            self.asset_proj = nn.Identity()
            asset_dim = enc_dim

        if asset_mixer_layers > 0:
            valid_heads = [h for h in [1, 2, 4, 8] if asset_dim % h == 0]
            if not valid_heads:
                asset_mixer_heads = 1
            elif asset_mixer_heads not in valid_heads:
                asset_mixer_heads = max(valid_heads)

        self.asset_mixer = CrossAssetMixer(
            d_model=asset_dim,
            num_heads=asset_mixer_heads if asset_mixer_layers > 0 else 1,
            dropout=dropout,
            num_layers=asset_mixer_layers,
        )

        self.use_residual_fusion = use_residual_fusion

        fusion_in = num_assets * asset_dim
        if fusion_hidden > 0:
            self.head = nn.Sequential(
                nn.Linear(fusion_in, fusion_hidden),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(fusion_hidden, num_assets),
            )
        else:
            self.head = nn.Linear(fusion_in, num_assets)

        if use_residual_fusion:
            self.residual_head = nn.Linear(asset_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, N, F = x.shape
        if N != self.num_assets:
            raise ValueError(f"Expected N={self.num_assets}, got N={N}")

        x = x.permute(0, 2, 1, 3).contiguous().reshape(B * N, L, F)
        out, _ = self.encoder(x)
        h = out[:, -1, :]
        h = self.asset_proj(h)
        h = h.reshape(B, N, -1)

        z_assets = self.asset_mixer(h)

        z = z_assets.reshape(B, -1)
        yhat = self.head(z)

        if self.use_residual_fusion:
            yhat = yhat + self.residual_head(z_assets).squeeze(-1)

        return yhat

## 10) Loss & metrics

In [ ]:
# Training objective: masked MSE or masked Huber loss (Optuna tunes which).
# Early stopping: on validation loss (not IC) to avoid gradient issues.
# Eval metrics: RMSE, MAE, R², directional accuracy, IC (Pearson + Spearman).
# Optuna objective: mean IC Spearman across folds × seeds.

def masked_mse_loss(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    mask = mask.float()
    target_safe = torch.where(mask > 0, target, torch.zeros_like(target))
    pred_safe = torch.where(mask > 0, pred, torch.zeros_like(pred))

    err2 = (pred_safe - target_safe) ** 2
    err2 = err2 * mask

    denom = torch.clamp(mask.sum(), min=1.0)
    loss = err2.sum() / denom

    if not torch.isfinite(loss):
        raise ValueError("masked_mse_loss became non-finite.")
    return loss


def masked_huber_loss(
    pred: torch.Tensor,
    target: torch.Tensor,
    mask: torch.Tensor,
    delta: float = 0.01
) -> torch.Tensor:
    mask = mask.float()
    target_safe = torch.where(mask > 0, target, torch.zeros_like(target))
    pred_safe = torch.where(mask > 0, pred, torch.zeros_like(pred))

    err = pred_safe - target_safe
    abs_err = torch.abs(err)

    delta_t = torch.tensor(delta, dtype=pred.dtype, device=pred.device)
    quad = torch.minimum(abs_err, delta_t)
    lin = abs_err - quad
    loss = 0.5 * quad ** 2 + delta_t * lin
    loss = loss * mask

    denom = torch.clamp(mask.sum(), min=1.0)
    loss = loss.sum() / denom

    if not torch.isfinite(loss):
        raise ValueError("masked_huber_loss became non-finite.")
    return loss


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray, mask: np.ndarray) -> Dict[str, float]:
    valid = mask.astype(bool)
    yt = y_true[valid]
    yp = y_pred[valid]

    if yt.size == 0:
        return {
            "rmse": np.nan,
            "mae": np.nan,
            "r2": np.nan,
            "directional_acc": np.nan,
            "ic_pearson": np.nan,
            "ic_spearman": np.nan,
        }

    rmse = float(np.sqrt(np.mean((yp - yt) ** 2)))
    mae = float(np.mean(np.abs(yp - yt)))

    ss_res = float(np.sum((yt - yp) ** 2))
    ss_tot = float(np.sum((yt - np.mean(yt)) ** 2))
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 1e-12 else np.nan

    directional_acc = float(np.mean(np.sign(yp) == np.sign(yt)))

    if yt.size > 1 and np.std(yt) > 1e-12 and np.std(yp) > 1e-12:
        ic_pearson = float(np.corrcoef(yt, yp)[0, 1])

        rank_yt = pd.Series(yt).rank().to_numpy()
        rank_yp = pd.Series(yp).rank().to_numpy()
        if np.std(rank_yt) > 1e-12 and np.std(rank_yp) > 1e-12:
            ic_spearman = float(np.corrcoef(rank_yt, rank_yp)[0, 1])
        else:
            ic_spearman = np.nan
    else:
        ic_pearson = np.nan
        ic_spearman = np.nan

    return {
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "directional_acc": directional_acc,
        "ic_pearson": ic_pearson,
        "ic_spearman": ic_spearman,
    }


def per_asset_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    mask: np.ndarray,
    asset_names: List[str]
) -> List[Dict[str, Any]]:
    rows = []
    for i, a in enumerate(asset_names):
        m = regression_metrics(y_true[:, [i]], y_pred[:, [i]], mask[:, [i]])
        rows.append({"asset": a, **m})
    return rows

## 11) Training loop

In [ ]:
def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    loss_name: str = "mse",
    huber_delta: float = 0.01,
) -> Dict[str, Any]:
    model.eval()

    ys, yhs, ms, ts = [], [], [], []
    losses = []

    for xb, yb, mb, tb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        mb = mb.to(device)

        pred = model(xb)

        if loss_name == "huber":
            loss = masked_huber_loss(pred, yb, mb, delta=huber_delta)
        else:
            loss = masked_mse_loss(pred, yb, mb)

        losses.append(float(loss.item()))
        ys.append(yb.detach().cpu().numpy())
        yhs.append(pred.detach().cpu().numpy())
        ms.append(mb.detach().cpu().numpy())
        ts.append(tb.detach().cpu().numpy())

    y_true = np.concatenate(ys, axis=0) if ys else np.empty((0, 0), dtype=np.float32)
    y_pred = np.concatenate(yhs, axis=0) if yhs else np.empty((0, 0), dtype=np.float32)
    mask = np.concatenate(ms, axis=0) if ms else np.empty((0, 0), dtype=np.float32)
    t_idx = np.concatenate(ts, axis=0) if ts else np.empty((0,), dtype=np.int64)

    metrics = regression_metrics(y_true, y_pred, mask)
    metrics["loss"] = float(np.mean(losses)) if losses else np.nan

    return {
        "metrics": metrics,
        "y_true": y_true,
        "y_pred": y_pred,
        "mask": mask,
        "t_idx": t_idx,
    }


def train_one_model(
    X: np.ndarray,
    y: np.ndarray,                 # training target (y_model)
    y_mask: np.ndarray,
    market: Optional[np.ndarray],
    train_t: np.ndarray,
    es_t: np.ndarray,
    lookback: int,
    input_dim_per_asset: int,
    output_dim: int,
    params: Dict[str, Any],
    seed: int,
    logger: logging.Logger,
) -> Tuple[nn.Module, Dict[str, Any]]:
    seed_everything(seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader = make_loader(
        X, y, y_mask, train_t, lookback, params["batch_size"], market, shuffle=True
    )
    es_loader = make_loader(
        X, y, y_mask, es_t, lookback, params["batch_size"], market, shuffle=False
    )

    model = JointAssetLSTM(
        input_dim_per_asset=input_dim_per_asset,
        num_assets=output_dim,
        hidden_size=params["hidden_size"],
        num_layers=params["num_layers"],
        dropout=params["dropout"],
        bidirectional=params["bidirectional"],
        encoder_proj=params["encoder_proj"],
        asset_mixer_layers=params["asset_mixer_layers"],
        asset_mixer_heads=params["asset_mixer_heads"],
        fusion_hidden=params["fusion_hidden"],
        use_residual_fusion=params["use_residual_fusion"],
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=params["learning_rate"],
        weight_decay=params["weight_decay"],
    )

    if params["scheduler"] == "cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max(params["max_epochs"], 5)
        )
    elif params["scheduler"] == "plateau":
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=3
        )
    else:
        scheduler = None

    best_state = None
    best_score = np.inf
    best_epoch = -1
    patience_count = 0

    hist = {"train_loss": [], "es_loss": [], "es_rmse": []}

    for epoch in range(1, params["max_epochs"] + 1):
        model.train()
        batch_losses = []

        for xb, yb, mb, _ in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            mb = mb.to(device)

            if not torch.isfinite(xb).all():
                raise ValueError("Non-finite xb detected.")
            if mb.sum() <= 0:
                continue
            if not torch.isfinite(yb[mb > 0]).all():
                raise ValueError("Non-finite valid targets detected.")
            if not torch.isfinite(mb).all():
                raise ValueError("Non-finite mask detected.")

            optimizer.zero_grad()
            pred = model(xb)

            if not torch.isfinite(pred).all():
                raise ValueError("Model prediction became non-finite.")

            if params["loss_name"] == "huber":
                loss = masked_huber_loss(pred, yb, mb, delta=params["huber_delta"])
            else:
                loss = masked_mse_loss(pred, yb, mb)

            if not torch.isfinite(loss):
                raise ValueError("Loss became non-finite before backward.")

            loss.backward()

            if params["grad_clip"] is not None and params["grad_clip"] > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), params["grad_clip"])

            optimizer.step()
            batch_losses.append(float(loss.item()))

        train_loss = float(np.mean(batch_losses)) if batch_losses else np.nan

        es_eval = evaluate_model(
            model, es_loader, device,
            loss_name=params["loss_name"],
            huber_delta=params["huber_delta"],
        )
        es_loss = float(es_eval["metrics"]["loss"])
        es_rmse = float(es_eval["metrics"]["rmse"])

        hist["train_loss"].append(train_loss)
        hist["es_loss"].append(es_loss)
        hist["es_rmse"].append(es_rmse)

        logger.info(
            f"[TRAIN] seed={seed} epoch={epoch:03d} "
            f"train_loss={train_loss:.6f} es_loss={es_loss:.6f} es_rmse={es_rmse:.6f}"
        )

        if scheduler is not None:
            if params["scheduler"] == "plateau":
                scheduler.step(es_loss)
            else:
                scheduler.step()

        improved = np.isfinite(es_loss) and es_loss < (best_score - params["min_delta"])
        if improved:
            best_score = es_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1

        if epoch >= params["min_epochs"] and patience_count >= params["patience"]:
            logger.info(
                f"[EARLY STOP] seed={seed} "
                f"best_epoch={best_epoch} best_es_loss={best_score:.6f}"
            )
            break

    if best_state is None:
        raise ValueError("No valid best_state found during training.")

    model.load_state_dict(best_state)
    return model, hist

## 12) Plotting & export

In [ ]:

def save_cumulative_returns_plot(
    out_dir: str,
    prefix: str,
    dates: pd.DatetimeIndex,
    y_true: np.ndarray,    # [T,N]
    y_pred: np.ndarray,    # [T,N]
    mask: np.ndarray,      # [T,N]
    asset_names: List[str],
):
    safe_makedirs(out_dir)

    plt.figure(figsize=(12, 6))

    any_line = False
    for i, asset in enumerate(asset_names):
        valid = mask[:, i].astype(bool)
        if valid.sum() == 0:
            continue

        yt = y_true[valid, i]
        yp = y_pred[valid, i]
        dt = dates[valid]

        strat_ret = np.sign(yp) * yt
        wealth = np.cumprod(1.0 + strat_ret)

        plt.plot(dt, wealth, label=asset)
        any_line = True

    if any_line:
        plt.title(f"{prefix} | cumulative returns by asset")
        plt.xlabel("date")
        plt.ylabel("wealth")
        plt.legend(loc="best", ncol=2)
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"{prefix}_cumulative_returns.png"))
    plt.close()


def save_per_asset_metrics_json(
    out_path: str,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    mask: np.ndarray,
    asset_names: List[str],
):
    metrics = per_asset_metrics(y_true, y_pred, mask, asset_names)
    with open(out_path, "w") as f:
        json.dump(metrics, f, indent=2)

## 13) One split / one seed

In [ ]:
# Called inside the Optuna objective for each (split, seed) combination.
# Handles: early-stop split, sample validity filter, model training, val eval, artifact saving.

def train_and_eval_one_split_seed(
    X_transformed: np.ndarray,
    y_model: np.ndarray,
    y_mask: np.ndarray,
    market_transformed: Optional[np.ndarray],
    asset_names: List[str],
    dates_index: pd.DatetimeIndex,
    train_idx: np.ndarray,
    val_idx: np.ndarray,
    lookback: int,
    seed: int,
    model_params: Dict[str, Any],
    trial_dir: str,
    logger: logging.Logger,
    split_i: int,
    save_artifacts: bool = True,
) -> Dict[str, Any]:
    train_core_idx, es_idx = split_train_for_early_stop(
        train_idx=train_idx,
        dates_index=dates_index,
        es_years=1,
        min_train_days=252,
    )

    sample_valid = compute_sample_validity(
        X_raw=X_transformed,
        y_mask=y_mask,
        lookback=lookback,
        min_valid_targets=max(1, len(asset_names) // 3),
    )

    train_t = train_core_idx[sample_valid[train_core_idx]]
    es_t = es_idx[sample_valid[es_idx]]
    val_t = val_idx[sample_valid[val_idx]]

    if len(train_t) < 64 or len(es_t) < 32 or len(val_t) < 32:
        raise optuna.exceptions.TrialPruned()

    if not np.isfinite(X_transformed).all():
        bad = np.argwhere(~np.isfinite(X_transformed))
        raise ValueError(f"X_transformed contains non-finite values. Example bad idx: {bad[:5]}")

    if market_transformed is not None and not np.isfinite(market_transformed).all():
        bad = np.argwhere(~np.isfinite(market_transformed))
        raise ValueError(f"market_transformed contains non-finite values. Example bad idx: {bad[:5]}")

    bad_targets = np.isfinite(y_model[train_t][y_mask[train_t]])
    if bad_targets.size > 0 and not bad_targets.all():
        raise ValueError("Valid training targets contain non-finite values.")

    input_dim_per_asset = X_transformed.shape[2] + (1 if market_transformed is not None else 0)
    output_dim = len(asset_names)

    logger.info(
        f"[SPLIT {split_i}] seed={seed} "
        f"train_samples={len(train_t)} es_samples={len(es_t)} val_samples={len(val_t)} "
        f"lookback={lookback} input_dim_per_asset={input_dim_per_asset} outputs={output_dim}"
    )

    model, _ = train_one_model(
        X=X_transformed,
        y=y_model,
        y_mask=y_mask,
        market=market_transformed,
        train_t=train_t,
        es_t=es_t,
        lookback=lookback,
        input_dim_per_asset=input_dim_per_asset,
        output_dim=output_dim,
        params=model_params,
        seed=seed,
        logger=logger,
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    val_loader = make_loader(
        X_transformed, y_model, y_mask, val_t, lookback,
        model_params["batch_size"], market_transformed, shuffle=False
    )
    val_eval = evaluate_model(
        model, val_loader, device,
        loss_name=model_params["loss_name"],
        huber_delta=model_params["huber_delta"],
    )

    val_metrics = val_eval["metrics"]
    logger.info(
        f"[VAL] split={split_i} seed={seed} "
        f"rmse={val_metrics['rmse']:.6f} mae={val_metrics['mae']:.6f} "
        f"r2={val_metrics['r2']:.6f} dir_acc={val_metrics['directional_acc']:.4f} "
        f"ic_p={val_metrics['ic_pearson']:.4f} ic_s={val_metrics['ic_spearman']:.4f}"
    )

    if save_artifacts:
        plot_dir = os.path.join(trial_dir, "artifacts", f"split_{split_i}", f"seed_{seed}")
        safe_makedirs(plot_dir)

        val_dates = dates_index[val_eval["t_idx"]]

        save_cumulative_returns_plot(
            out_dir=plot_dir,
            prefix=f"split{split_i}_seed{seed}_val",
            dates=val_dates,
            y_true=val_eval["y_true"],
            y_pred=val_eval["y_pred"],
            mask=val_eval["mask"],
            asset_names=asset_names,
        )

        save_per_asset_metrics_json(
            out_path=os.path.join(plot_dir, f"split{split_i}_seed{seed}_per_asset_metrics.json"),
            y_true=val_eval["y_true"],
            y_pred=val_eval["y_pred"],
            mask=val_eval["mask"],
            asset_names=asset_names,
        )

    return {
        "metrics": val_metrics,
        "y_true": val_eval["y_true"],
        "y_pred": val_eval["y_pred"],
        "mask": val_eval["mask"],
        "t_idx": val_eval["t_idx"],
    }

## 14) Optuna objective & study runner

In [ ]:
# objective_factory closes over the dataset and splits.
# Each trial: sample features + architecture + training params → train 3 folds × 3 seeds → mean IC Spearman.
# Optuna MedianPruner drops weak trials after the first fold.

def objective_factory(
    df: pd.DataFrame,
    assets: List[str],
    features: Dict[str, List[str]],
    base_log_dir: str,
    seeds: List[int],
    spreads: Optional[np.ndarray] = None,
):
    dates_index = pd.DatetimeIndex(df.index)
    splits = make_walk_forward_splits(dates_index)

    def objective(trial: optuna.Trial) -> float:
        trial_dir = os.path.join(base_log_dir, f"trial_{trial.number:05d}")
        #os.makedirs(trial_dir, exist_ok=True)
        logger = setup_logger(trial_dir)

        logger.info("=" * 100)
        logger.info(f"TRIAL {trial.number} START")

        picks = pick_features_from_buckets(trial, features)
        per_asset_feats = get_per_asset_feature_list(picks)

        trial.set_user_attr("feature_set", picks)
        trial.set_user_attr("per_asset_feats", per_asset_feats)

        lookback = trial.suggest_categorical("lookback", [5, 10, 22])

        hidden_size = trial.suggest_categorical("hidden_size", [32, 64, 128])
        num_layers = trial.suggest_int("num_layers", 1, 2)
        dropout = trial.suggest_float("dropout", 0.0, 0.3)
        bidirectional = bool(trial.suggest_categorical("bidirectional", [0, 1]))
        encoder_proj = trial.suggest_categorical("encoder_proj", [0, 32, 64])

        asset_mixer_layers = trial.suggest_categorical("asset_mixer_layers", [0, 1, 2])
        asset_mixer_heads = trial.suggest_categorical("asset_mixer_heads", [1, 2, 4])
        fusion_hidden = trial.suggest_categorical("fusion_hidden", [0, 64, 128])
        use_residual_fusion = bool(trial.suggest_categorical("use_residual_fusion", [0, 1]))

        scheduler = trial.suggest_categorical("scheduler", ["none", "plateau"])
        learning_rate = trial.suggest_float("learning_rate", 1e-4, 2e-3, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
        batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
        max_epochs = trial.suggest_categorical("max_epochs", [20, 30, 40])
        patience = trial.suggest_categorical("patience", [5, 7])
        min_epochs = trial.suggest_categorical("min_epochs", [5, 8])
        grad_clip = trial.suggest_categorical("grad_clip", [0.5, 1.0, 2.0])

        loss_name = trial.suggest_categorical("loss_name", ["mse", "huber"])
        huber_delta = trial.suggest_categorical("huber_delta", [0.005, 0.01, 0.02])
        min_delta = trial.suggest_categorical("min_delta", [0.0, 1e-5])

        model_params = {
            "hidden_size": hidden_size,
            "num_layers": num_layers,
            "dropout": dropout,
            "bidirectional": bidirectional,
            "encoder_proj": encoder_proj,
            "asset_mixer_layers": asset_mixer_layers,
            "asset_mixer_heads": asset_mixer_heads,
            "fusion_hidden": fusion_hidden,
            "use_residual_fusion": use_residual_fusion,
            "scheduler": scheduler,
            "learning_rate": learning_rate,
            "weight_decay": weight_decay,
            "batch_size": batch_size,
            "max_epochs": max_epochs,
            "patience": patience,
            "min_epochs": min_epochs,
            "grad_clip": grad_clip,
            "loss_name": loss_name,
            "huber_delta": huber_delta,
            "min_delta": min_delta,
        }

        logger.info(f"Feature picks: {picks}")
        logger.info(f"Per-asset feats: {per_asset_feats}")
        logger.info(f"Model params: {model_params}")
        logger.info(f"lookback={lookback}")

        _, X_raw, y_model, y_raw, y_mask, feat_names, market_raw = build_raw_arrays(
            df=df,
            assets=assets,
            picks=picks,
            features=features,
            logger=logger,
            spreads=spreads,
        )

        all_rmse = []
        all_ic = []
        all_scores = []

        for split_i, (train_idx, val_idx) in enumerate(splits):
            logger.info("-" * 100)
            logger.info(
                f"[SPLIT {split_i}] "
                f"train {df.index[train_idx[0]]}..{df.index[train_idx[-1]]} | "
                f"val {df.index[val_idx[0]]}..{df.index[val_idx[-1]]}"
            )

            X_train_raw = X_raw[train_idx]
            market_train_raw = market_raw[train_idx] if market_raw is not None else None

            fitted, market_ft = fit_transforms_on_train(X_train_raw, feat_names, market_train_raw)
            X_trans, market_trans = apply_transforms(X_raw, fitted, market_raw, market_ft)

            split_seed_scores = []

            for seed in seeds:
                try:
                    res = train_and_eval_one_split_seed(
                        X_transformed=X_trans,
                        y_model=y_model,
                        y_mask=y_mask,
                        market_transformed=market_trans,
                        asset_names=assets,
                        dates_index=dates_index,
                        train_idx=train_idx,
                        val_idx=val_idx,
                        lookback=lookback,
                        seed=seed,
                        model_params=model_params,
                        trial_dir=trial_dir,
                        logger=logger,
                        split_i=split_i,
                        save_artifacts=False,
                    )
                except ValueError as e:
                    logger.info(f"[PRUNE] Non-finite training trial: {e}")
                    raise optuna.TrialPruned()

                rmse = float(res["metrics"]["rmse"])
                ic_spearman = float(res["metrics"]["ic_spearman"]) if np.isfinite(res["metrics"]["ic_spearman"]) else -999.0

                score = ic_spearman

                all_rmse.append(rmse)
                all_ic.append(ic_spearman)
                all_scores.append(score)
                split_seed_scores.append(score)

                logger.info(
                    f"[SPLIT {split_i}][SEED {seed}] "
                    f"score={score:.6f} rmse={rmse:.6f} ic_spearman={ic_spearman:.6f}"
                )

            split_mean = float(np.mean(split_seed_scores))
            trial.report(split_mean, step=split_i)
            if trial.should_prune():
                logger.info("[PRUNE] Optuna pruned this trial.")
                raise optuna.TrialPruned()

        final_score = float(np.mean(all_scores))

        trial.set_user_attr("all_scores", [float(x) for x in all_scores])
        trial.set_user_attr("mean_rmse", float(np.mean(all_rmse)))
        trial.set_user_attr("mean_ic", float(np.mean(all_ic)))
        trial.set_user_attr("note", "Model trained/evaluated on spread-adjusted y_model; BL/backtest uses raw y_raw.")

        logger.info("=" * 100)
        logger.info(
            f"TRIAL {trial.number} DONE | "
            f"score={final_score:.6f} mean_rmse={np.mean(all_rmse):.6f} mean_ic={np.mean(all_ic):.6f}"
        )

        return final_score

    return objective



def run_optuna_search(
    df: pd.DataFrame,
    assets: List[str],
    features: Dict[str, List[str]],
    out_dir: str,
    n_trials: int = 50,
    seeds: Optional[List[int]] = None,
    spreads: Optional[np.ndarray] = None,
):
    os.makedirs(out_dir, exist_ok=True)
    seeds = seeds or [11, 22, 33]

    storage_path = os.path.join(out_dir, "joint_asset_lstm.db")
    storage = f"sqlite:///{storage_path}"

    study = optuna.create_study(
        study_name="joint_asset_lstm",
        direction="maximize",
        storage=storage,
        load_if_exists=True,
        sampler=TPESampler(seed=None, multivariate=True, group=True, constant_liar=True),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1),
    )

    objective = objective_factory(
        df=df,
        assets=assets,
        features=features,
        base_log_dir=out_dir,
        seeds=seeds,
        spreads=spreads,
    )

    study.optimize(objective, n_trials=n_trials, gc_after_trial=True, show_progress_bar=True)

    best = study.best_trial
    summary = {
        "best_value": best.value,
        "best_params": best.params,
        "best_user_attrs": best.user_attrs,
        "n_trials": len(study.trials),
    }

    with open(os.path.join(out_dir, "best_trial.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print("\n=== BEST TRIAL ===")
    print("Value:", best.value)
    print("Params:", best.params)
    print("Feature set:", best.user_attrs.get("feature_set"))
    print("Per-asset feats:", best.user_attrs.get("per_asset_feats"))
    print("Mean RMSE:", best.user_attrs.get("mean_rmse"))
    print("Mean IC:", best.user_attrs.get("mean_ic"))
    print("DB:", storage_path)
    return study, out_dir

## 15) Search space inspector

In [ ]:
def estimate_search_space(study: optuna.Study) -> int:
    """
    Inspect the study's distributions and print a suggested N_TRIALS ceiling.

    Discrete combinations (categorical × int params) are counted exactly.
    Continuous params (float) are infinite and excluded from the count.

    Suggested ceiling = ceil(sqrt(discrete_combos))
    The ×5 factor accounts for continuous params and seed variance.
    Nothing is applied automatically — use the output to guide N_TRIALS.
    """
    if not study.trials:
        raise ValueError('Run at least 1 trial first so distributions are populated.')

    distributions     = study.trials[0].distributions
    discrete_combinations = 1
    continuous_params = []

    print('Search space:')
    for name, dist in distributions.items():
        if isinstance(dist, optuna.distributions.CategoricalDistribution):
            n = len(dist.choices)
            discrete_combinations *= n
            print(f'  {name:30s}  categorical  {n:>4d} choices')
        elif isinstance(dist, optuna.distributions.IntDistribution):
            n = (dist.high - dist.low) // dist.step + 1
            discrete_combinations *= n
            print(f'  {name:30s}  int          {n:>4d} levels  [{dist.low} .. {dist.high}]')
        elif isinstance(dist, optuna.distributions.FloatDistribution):
            continuous_params.append(name)
            scale = 'log' if dist.log else 'linear'
            print(f'  {name:30s}  float ({scale:6s})       [{dist.low:.2e} .. {dist.high:.2e}]  ← continuous, excluded')

    import math
    suggested = math.ceil(math.sqrt(discrete_combinations))
    print()
    print(f'Discrete combinations  : {discrete_combinations}')
    print(f'Continuous params      : {continuous_params}')
    print(f'Suggested N_TRIALS     : ceil(sqrt({discrete_combinations})) = {suggested}')
    print(f'  └─ treat this as a ceiling; more trials risk overfitting the val folds')
    return suggested


## 16) Refit best trial

In [ ]:
# fit_best_trial_and_export refits the best trial's params on each walk-forward split
# and exports predictions + metrics as CSVs. Used to inspect in-sample/OOS performance
# and to feed the ensemble into the Black-Litterman step.

def pick_features_from_best_params(best_params: Dict[str, Any], features: Dict[str, List[str]]) -> Dict[str, Any]:
    class DummyTrial:
        def __init__(self, params):
            self.params = params

        def suggest_categorical(self, name, choices):
            return self.params[name]

    dummy = DummyTrial(best_params)
    return pick_features_from_buckets(dummy, features)


def fit_best_trial_and_export(
    df: pd.DataFrame,
    assets: List[str],
    features: Dict[str, List[str]],
    study: optuna.Study,
    out_dir: str,
    seeds: Optional[List[int]] = None,
    spreads: Optional[np.ndarray] = None,
):
    seeds = seeds or [11, 22, 33]
    best = study.best_trial
    logger = setup_logger(os.path.join(out_dir, "best_refit"))

    picks = pick_features_from_best_params(best.params, features)
    lookback = best.params["lookback"]

    model_params = {
        "hidden_size": best.params["hidden_size"],
        "num_layers": best.params["num_layers"],
        "dropout": best.params["dropout"],
        "bidirectional": bool(best.params["bidirectional"]),
        "encoder_proj": best.params["encoder_proj"],
        "asset_mixer_layers": best.params["asset_mixer_layers"],
        "asset_mixer_heads": best.params["asset_mixer_heads"],
        "fusion_hidden": best.params["fusion_hidden"],
        "use_residual_fusion": bool(best.params["use_residual_fusion"]),
        "scheduler": best.params["scheduler"],
        "learning_rate": best.params["learning_rate"],
        "weight_decay": best.params["weight_decay"],
        "batch_size": best.params["batch_size"],
        "max_epochs": best.params["max_epochs"],
        "patience": best.params["patience"],
        "min_epochs": best.params["min_epochs"],
        "grad_clip": best.params["grad_clip"],
        "loss_name": best.params["loss_name"],
        "huber_delta": best.params["huber_delta"],
        "min_delta": best.params["min_delta"],
    }

    dates_index = pd.DatetimeIndex(df.index)
    splits = make_walk_forward_splits(dates_index)

    _, X_raw, y_model, y_raw, y_mask, feat_names, market_raw = build_raw_arrays(
        df=df,
        assets=assets,
        picks=picks,
        features=features,
        logger=logger,
        spreads=spreads,
    )

    rows = []

    for split_i, (train_idx, val_idx) in enumerate(splits):
        X_train_raw = X_raw[train_idx]
        market_train_raw = market_raw[train_idx] if market_raw is not None else None
        fitted, market_ft = fit_transforms_on_train(X_train_raw, feat_names, market_train_raw)
        X_trans, market_trans = apply_transforms(X_raw, fitted, market_raw, market_ft)

        for seed in seeds:
            res = train_and_eval_one_split_seed(
                X_transformed=X_trans,
                y_model=y_model,
                y_mask=y_mask,
                market_transformed=market_trans,
                asset_names=assets,
                dates_index=dates_index,
                train_idx=train_idx,
                val_idx=val_idx,
                lookback=lookback,
                seed=seed,
                model_params=model_params,
                trial_dir=os.path.join(out_dir, "best_refit"),
                logger=logger,
                split_i=split_i,
                save_artifacts=False,
            )

            rows.append({
                "split": split_i,
                "seed": seed,
                **res["metrics"],
            })

    summary_df = pd.DataFrame(rows)
    summary_df.to_csv(os.path.join(out_dir, "best_refit", "summary_metrics.csv"), index=False)
    print(summary_df.groupby("split").mean(numeric_only=True))
    return summary_df

## 17) Black–Litterman portfolio construction

In [ ]:
# Black-Litterman workflow:
#   1. Ensemble predictions from top-N Optuna trials → views vector P·μ
#   2. Covariance estimated from trailing training window → Σ
#   3. BL posterior: combine market equilibrium prior with LSTM views
#   4. Mean-variance optimisation on BL posterior → portfolio weights
#
# BL hyperparameters (tau, risk_aversion, view_confidence) are tuned by a
# second Optuna study that maximises walk-forward validation Sharpe.

def _safe_float(x):
    try:
        v = float(x)
        if np.isfinite(v):
            return v
        return None
    except Exception:
        return None


def get_top_n_completed_trials(study: optuna.Study, top_n: int = 5) -> List[optuna.trial.FrozenTrial]:
    trials = [
        t for t in study.trials
        if t.state == TrialState.COMPLETE and t.value is not None and np.isfinite(t.value)
    ]
    trials = sorted(trials, key=lambda t: t.value, reverse=True)
    if len(trials) == 0:
        raise ValueError("No completed Optuna trials found.")
    return trials[:top_n]


def make_final_train_test_split_by_target_date(
    dates_index: pd.DatetimeIndex,
    train_target_end: str = "2023-06-30",
    test_target_start: str = "2023-07-01",
) -> Tuple[np.ndarray, np.ndarray]:
    """
    IMPORTANT:
    y[t] is the next-day return from dates[t] -> dates[t+1].
    So we split using the TARGET date = dates[t+1], not the anchor date = dates[t].

    train_idx: anchor indices t where target date <= train_target_end
    test_idx : anchor indices t where target date >= test_target_start
    """
    if len(dates_index) < 2:
        raise ValueError("Need at least 2 rows in df index.")

    anchor_idx = np.arange(len(dates_index) - 1, dtype=np.int64)
    target_dates = dates_index[1:]

    train_end = pd.Timestamp(train_target_end)
    test_start = pd.Timestamp(test_target_start)

    train_idx = anchor_idx[target_dates <= train_end]
    test_idx = anchor_idx[target_dates >= test_start]

    if len(train_idx) == 0 or len(test_idx) == 0:
        raise ValueError(
            f"Empty final split. train_idx={len(train_idx)} test_idx={len(test_idx)}"
        )

    return train_idx, test_idx


def build_model_params_from_best_params(best_params: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "hidden_size": best_params["hidden_size"],
        "num_layers": best_params["num_layers"],
        "dropout": best_params["dropout"],
        "bidirectional": bool(best_params["bidirectional"]),
        "encoder_proj": best_params["encoder_proj"],
        "asset_mixer_layers": best_params["asset_mixer_layers"],
        "asset_mixer_heads": best_params["asset_mixer_heads"],
        "fusion_hidden": best_params["fusion_hidden"],
        "use_residual_fusion": bool(best_params["use_residual_fusion"]),
        "scheduler": best_params["scheduler"],
        "learning_rate": best_params["learning_rate"],
        "weight_decay": best_params["weight_decay"],
        "batch_size": best_params["batch_size"],
        "max_epochs": best_params["max_epochs"],
        "patience": best_params["patience"],
        "min_epochs": best_params["min_epochs"],
        "grad_clip": best_params["grad_clip"],
        "loss_name": best_params["loss_name"],
        "huber_delta": best_params["huber_delta"],
        "min_delta": best_params["min_delta"],
    }


def evaluate_model_on_t_indices(
    model: nn.Module,
    X: np.ndarray,
    y: np.ndarray,
    y_mask: np.ndarray,
    market: Optional[np.ndarray],
    t_idx: np.ndarray,
    lookback: int,
    batch_size: int,
    loss_name: str,
    huber_delta: float,
) -> Dict[str, Any]:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    loader = make_loader(
        X=X,
        y=y,
        y_mask=y_mask,
        valid_t=t_idx,
        lookback=lookback,
        batch_size=batch_size,
        market=market,
        shuffle=False,
    )
    return evaluate_model(
        model=model,
        loader=loader,
        device=device,
        loss_name=loss_name,
        huber_delta=huber_delta,
    )


def _full_pred_panel_from_eval(
    T: int,
    N: int,
    eval_res: Dict[str, Any],
) -> Tuple[np.ndarray, np.ndarray]:
    pred_full = np.full((T, N), np.nan, dtype=np.float32)
    obs_mask = np.zeros((T, N), dtype=bool)

    if len(eval_res["t_idx"]) > 0:
        pred_full[eval_res["t_idx"]] = eval_res["y_pred"]
        obs_mask[eval_res["t_idx"]] = eval_res["mask"].astype(bool)

    return pred_full, obs_mask


def nanmean_predictions(pred_list: List[np.ndarray]) -> np.ndarray:
    stack = np.stack(pred_list, axis=0)
    return np.nanmean(stack, axis=0).astype(np.float32)


def estimate_train_covariance(
    y_train: np.ndarray,          # raw realized returns [T,N]
    y_mask_train: np.ndarray,     # [T,N]
    ridge: float = 1e-5,
) -> np.ndarray:
    y_df = pd.DataFrame(np.where(y_mask_train, y_train, np.nan))
    Sigma = y_df.cov(min_periods=max(20, len(y_df) // 10)).to_numpy(dtype=np.float64)

    if Sigma.shape[0] != y_train.shape[1]:
        raise ValueError("Covariance shape mismatch.")

    Sigma = np.nan_to_num(Sigma, nan=0.0, posinf=0.0, neginf=0.0)

    diag = np.diag(Sigma).copy()
    pos_diag = diag[np.isfinite(diag) & (diag > 0)]
    fill_var = float(np.median(pos_diag)) if len(pos_diag) > 0 else 1e-4

    for i in range(len(diag)):
        if not np.isfinite(Sigma[i, i]) or Sigma[i, i] <= 0:
            Sigma[i, i] = fill_var

    Sigma = 0.5 * (Sigma + Sigma.T)
    Sigma = Sigma + ridge * np.eye(Sigma.shape[0], dtype=np.float64)
    return Sigma


def black_litterman_posterior(
    Sigma: np.ndarray,
    q: np.ndarray,
    market_weights: Optional[np.ndarray] = None,
    tau: float = 0.05,
    risk_aversion: float = 2.5,
    omega_scale: float = 0.25,
    ridge: float = 1e-8,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    P = I, Q = predicted returns.
    Prior equilibrium excess returns pi = risk_aversion * Sigma @ w_mkt
    """
    N = Sigma.shape[0]
    Sigma = np.asarray(Sigma, dtype=np.float64)
    Sigma = 0.5 * (Sigma + Sigma.T) + ridge * np.eye(N, dtype=np.float64)

    if market_weights is None:
        w_mkt = np.full(N, 1.0 / N, dtype=np.float64)
    else:
        w_mkt = np.asarray(market_weights, dtype=np.float64)
        w_mkt = np.clip(w_mkt, 0.0, None)
        if w_mkt.sum() <= 0:
            w_mkt = np.full(N, 1.0 / N, dtype=np.float64)
        else:
            w_mkt = w_mkt / w_mkt.sum()

    q = np.asarray(q, dtype=np.float64)

    pi = risk_aversion * (Sigma @ w_mkt)
    P = np.eye(N, dtype=np.float64)

    tauSigma = tau * Sigma
    omega_diag = np.maximum(np.diag(tauSigma) * omega_scale, ridge)
    Omega = np.diag(omega_diag)

    inv_tauSigma = np.linalg.inv(tauSigma)
    inv_Omega = np.linalg.inv(Omega)

    A = inv_tauSigma + P.T @ inv_Omega @ P
    b = inv_tauSigma @ pi + P.T @ inv_Omega @ q

    mu_bl = np.linalg.solve(A, b)
    return mu_bl, pi, Omega


def mean_variance_weights(
    mu: np.ndarray,
    Sigma: np.ndarray,
    risk_aversion: float = 2.5,
    long_only: bool = True,
    max_weight: Optional[float] = None,
    ridge: float = 1e-8,
) -> np.ndarray:
    A = risk_aversion * Sigma + ridge * np.eye(len(mu), dtype=np.float64)

    try:
        w = np.linalg.solve(A, mu)
    except np.linalg.LinAlgError:
        w = np.full(len(mu), 1.0 / len(mu), dtype=np.float64)

    if long_only:
        w = np.clip(w, 0.0, None)

    if max_weight is not None and max_weight > 0:
        w = np.minimum(w, max_weight)

    if w.sum() <= 0:
        w = np.full(len(mu), 1.0 / len(mu), dtype=np.float64)
    else:
        w = w / w.sum()

    return w


def portfolio_metrics_from_returns(rets: np.ndarray, freq: int = 252) -> Dict[str, float]:
    rets = np.asarray(rets, dtype=np.float64)
    rets = rets[np.isfinite(rets)]

    if rets.size == 0:
        return {
            "n_days": 0,
            "mean_daily_return": np.nan,
            "vol_daily": np.nan,
            "ann_return": np.nan,
            "ann_vol": np.nan,
            "sharpe": np.nan,
            "cum_return": np.nan,
            "max_drawdown": np.nan,
        }

    wealth = np.cumprod(1.0 + rets)
    cum_return = float(wealth[-1] - 1.0)
    mean_daily = float(np.mean(rets))
    vol_daily = float(np.std(rets, ddof=0))
    ann_return = float((1.0 + mean_daily) ** freq - 1.0)
    ann_vol = float(vol_daily * np.sqrt(freq))
    sharpe = float(mean_daily / vol_daily * np.sqrt(freq)) if vol_daily > 1e-12 else np.nan

    peak = np.maximum.accumulate(wealth)
    dd = wealth / peak - 1.0
    max_dd = float(np.min(dd))

    return {
        "n_days": int(rets.size),
        "mean_daily_return": mean_daily,
        "vol_daily": vol_daily,
        "ann_return": ann_return,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "cum_return": cum_return,
        "max_drawdown": max_dd,
    }


def run_black_litterman_backtest(
    dates_target: pd.DatetimeIndex,
    y_true: np.ndarray,                 # raw realized returns [T,N]
    y_pred: np.ndarray,                 # model predictions [T,N]
    mask: np.ndarray,                   # [T,N]
    asset_names: List[str],
    train_cov: np.ndarray,
    tau: float = 0.05,
    risk_aversion: float = 2.5,
    omega_scale: float = 0.25,
    long_only: bool = True,
    max_weight: Optional[float] = None,
) -> Dict[str, Any]:
    T, N = y_true.shape
    weights_full = np.zeros((T, N), dtype=np.float64)
    port_rets = np.full(T, np.nan, dtype=np.float64)

    for t in range(T):
        active = mask[t].astype(bool) & np.isfinite(y_pred[t]) & np.isfinite(y_true[t])
        if active.sum() == 0:
            continue

        idx = np.where(active)[0]
        q = y_pred[t, idx].astype(np.float64)
        Sigma_sub = train_cov[np.ix_(idx, idx)]

        mu_bl, _, _ = black_litterman_posterior(
            Sigma=Sigma_sub,
            q=q,
            market_weights=None,
            tau=tau,
            risk_aversion=risk_aversion,
            omega_scale=omega_scale,
        )

        w_sub = mean_variance_weights(
            mu=mu_bl,
            Sigma=Sigma_sub,
            risk_aversion=risk_aversion,
            long_only=long_only,
            max_weight=max_weight,
        )

        w = np.zeros(N, dtype=np.float64)
        w[idx] = w_sub
        weights_full[t] = w

        realized = y_true[t].astype(np.float64)
        port_rets[t] = float(np.dot(w, np.nan_to_num(realized, nan=0.0)))

    weights_df = pd.DataFrame(weights_full, index=dates_target, columns=asset_names)
    returns_df = pd.DataFrame(
        {
            "portfolio_return": port_rets,
            "portfolio_wealth": np.cumprod(1.0 + np.nan_to_num(port_rets, nan=0.0)),
        },
        index=dates_target,
    )

    if len(weights_df) > 1:
        turnover = np.abs(weights_df.diff().fillna(0.0)).sum(axis=1)
        avg_turnover = float(turnover.mean())
    else:
        avg_turnover = np.nan

    pm = portfolio_metrics_from_returns(port_rets[np.isfinite(port_rets)])
    pm["avg_daily_turnover"] = avg_turnover

    return {
        "metrics": pm,
        "weights_df": weights_df,
        "returns_df": returns_df,
    }


def make_anchor_idx_by_target_date_range(
    dates_index: pd.DatetimeIndex,
    target_start: Optional[str] = None,
    target_end: Optional[str] = None,
) -> np.ndarray:
    if len(dates_index) < 2:
        raise ValueError("Need at least 2 rows in dates_index.")

    anchor_idx = np.arange(len(dates_index) - 1, dtype=np.int64)
    target_dates = dates_index[1:]

    mask = np.ones(len(anchor_idx), dtype=bool)

    if target_start is not None:
        mask &= target_dates >= pd.Timestamp(target_start)
    if target_end is not None:
        mask &= target_dates <= pd.Timestamp(target_end)

    out = anchor_idx[mask]
    if len(out) == 0:
        raise ValueError(
            f"No anchors found for target date range start={target_start} end={target_end}"
        )
    return out


def refit_topn_and_predict_block(
    df: pd.DataFrame,
    assets: List[str],
    features: Dict[str, List[str]],
    top_trials: List[optuna.trial.FrozenTrial],
    train_idx: np.ndarray,
    test_idx: np.ndarray,
    dates_index: pd.DatetimeIndex,
    seed: int,
    logger: logging.Logger,
    spreads: Optional[np.ndarray] = None,
) -> Dict[str, Any]:
    T = len(df)
    N = len(assets)

    trial_full_preds = []
    canonical_y_raw = None
    canonical_y_mask = None

    for rank, trial in enumerate(top_trials, start=1):
        picks = pick_features_from_best_params(trial.params, features)
        lookback = int(trial.params["lookback"])
        model_params = build_model_params_from_best_params(trial.params)

        _, X_raw, y_model, y_raw, y_mask, feat_names, market_raw = build_raw_arrays(
            df=df,
            assets=assets,
            picks=picks,
            features=features,
            logger=logger,
            spreads=spreads,
        )

        if canonical_y_raw is None:
            canonical_y_raw = y_raw.copy()
            canonical_y_mask = y_mask.copy()

        X_train_raw = X_raw[train_idx]
        market_train_raw = market_raw[train_idx] if market_raw is not None else None
        fitted, market_ft = fit_transforms_on_train(X_train_raw, feat_names, market_train_raw)
        X_trans, market_trans = apply_transforms(X_raw, fitted, market_raw, market_ft)

        sample_valid = compute_sample_validity(
            X_raw=X_trans,
            y_mask=y_mask,
            lookback=lookback,
            min_valid_targets=max(1, len(assets) // 3),
        )

        train_core_idx, es_idx = split_train_for_early_stop(
            train_idx=train_idx,
            dates_index=dates_index,
            es_years=1,
            min_train_days=252,
        )

        train_core_t = train_core_idx[sample_valid[train_core_idx]]
        es_t = es_idx[sample_valid[es_idx]]
        test_t = test_idx[sample_valid[test_idx]]

        if len(train_core_t) < 64 or len(es_t) < 32 or len(test_t) < 32:
            logger.warning(
                f"Skipping top trial {trial.number}: "
                f"train_core={len(train_core_t)} es={len(es_t)} test={len(test_t)}"
            )
            continue

        input_dim_per_asset = X_trans.shape[2] + (1 if market_trans is not None else 0)

        model, _ = train_one_model(
            X=X_trans,
            y=y_model,
            y_mask=y_mask,
            market=market_trans,
            train_t=train_core_t,
            es_t=es_t,
            lookback=lookback,
            input_dim_per_asset=input_dim_per_asset,
            output_dim=len(assets),
            params=model_params,
            seed=seed,
            logger=logger,
        )

        test_eval = evaluate_model_on_t_indices(
            model=model,
            X=X_trans,
            y=y_model,
            y_mask=y_mask,
            market=market_trans,
            t_idx=test_t,
            lookback=lookback,
            batch_size=model_params["batch_size"],
            loss_name=model_params["loss_name"],
            huber_delta=model_params["huber_delta"],
        )

        pred_full = np.full((T, N), np.nan, dtype=np.float32)
        pred_full[test_eval["t_idx"]] = test_eval["y_pred"]
        trial_full_preds.append(pred_full)

    if len(trial_full_preds) == 0:
        raise ValueError("No usable top-N trials for this block.")

    ensemble_pred_full = nanmean_predictions(trial_full_preds)

    test_pred_mask = canonical_y_mask[test_idx] & np.isfinite(ensemble_pred_full[test_idx])

    train_cov = estimate_train_covariance(
        y_train=canonical_y_raw[train_idx],
        y_mask_train=canonical_y_mask[train_idx],
        ridge=1e-5,
    )

    return {
        "ensemble_pred_full": ensemble_pred_full,
        "y_raw": canonical_y_raw,
        "y_mask": canonical_y_mask,
        "test_pred_mask": test_pred_mask,
        "train_cov": train_cov,
    }


def _records_from_df(df: pd.DataFrame, date_col_name: str = "date") -> List[Dict[str, Any]]:
    out = []
    tmp = df.copy()
    tmp = tmp.reset_index()
    tmp = tmp.rename(columns={tmp.columns[0]: date_col_name})

    for row in tmp.to_dict(orient="records"):
        rec = {}
        for k, v in row.items():
            if isinstance(v, (pd.Timestamp, np.datetime64)):
                rec[k] = pd.Timestamp(v).strftime("%Y-%m-%d")
            elif isinstance(v, (np.floating, float, int, np.integer)):
                rec[k] = _safe_float(v) if not isinstance(v, (int, np.integer)) else int(v)
            else:
                rec[k] = v
        out.append(rec)
    return out


def final_topn_black_litterman_eval_with_midtest_refit(
    df: pd.DataFrame,
    assets: List[str],
    features: Dict[str, List[str]],
    study: optuna.Study,
    out_dir: str,
    top_n: int = 5,
    seed: int = SEED_PORTFOLIO_LSTM_RETURNS_V5_2_2,
    train_target_end: str = "2023-06-30",
    test_year1_start: str = "2023-07-01",
    test_year1_end: str = "2024-06-30",
    test_year2_start: str = "2024-07-01",
    test_year2_end: str = "2025-06-30",
    tau: float = 0.05,
    risk_aversion: float = 2.5,
    omega_scale: float = 0.25,
    long_only: bool = True,
    max_weight: Optional[float] = 0.35,
    bl_best_params: Optional[Dict[str, Any]] = None,
    spreads: Optional[np.ndarray] = None,
) -> Dict[str, Any]:
    final_dir = os.path.join(out_dir, "final_topn_black_litterman_midtest_refit")
    safe_makedirs(final_dir)
    logger = setup_logger(final_dir)

    if bl_best_params is not None:
        tau = float(bl_best_params.get("tau", tau))
        risk_aversion = float(bl_best_params.get("risk_aversion", risk_aversion))
        omega_scale = float(bl_best_params.get("omega_scale", omega_scale))
        max_weight = (
            None if bl_best_params.get("max_weight", max_weight) is None
            else float(bl_best_params.get("max_weight", max_weight))
        )
        long_only = bool(bl_best_params.get("long_only", long_only))
        top_n = int(bl_best_params.get("top_n", top_n))

    dates_index = pd.DatetimeIndex(df.index)
    top_trials = get_top_n_completed_trials(study, top_n=top_n)

    block1_train_idx = make_anchor_idx_by_target_date_range(
        dates_index, target_start=None, target_end=train_target_end
    )
    block1_test_idx = make_anchor_idx_by_target_date_range(
        dates_index, target_start=test_year1_start, target_end=test_year1_end
    )

    block2_train_idx = make_anchor_idx_by_target_date_range(
        dates_index, target_start=None, target_end=test_year1_end
    )
    block2_test_idx = make_anchor_idx_by_target_date_range(
        dates_index, target_start=test_year2_start, target_end=test_year2_end
    )

    logger.info("=" * 100)
    logger.info("FINAL TEST WITH MID-TEST REFIT")
    logger.info(
        f"Block1 train target end={train_target_end} | "
        f"Block1 test={test_year1_start}..{test_year1_end}"
    )
    logger.info(
        f"Block2 train target end={test_year1_end} | "
        f"Block2 test={test_year2_start}..{test_year2_end}"
    )

    block1 = refit_topn_and_predict_block(
        df=df,
        assets=assets,
        features=features,
        top_trials=top_trials,
        train_idx=block1_train_idx,
        test_idx=block1_test_idx,
        dates_index=dates_index,
        seed=seed,
        logger=logger,
        spreads=spreads,
    )

    block1_bl = run_black_litterman_backtest(
        dates_target=dates_index[block1_test_idx + 1],
        y_true=block1["y_raw"][block1_test_idx],
        y_pred=block1["ensemble_pred_full"][block1_test_idx],
        mask=block1["test_pred_mask"],
        asset_names=assets,
        train_cov=block1["train_cov"],
        tau=tau,
        risk_aversion=risk_aversion,
        omega_scale=omega_scale,
        long_only=long_only,
        max_weight=max_weight,
    )

    block2 = refit_topn_and_predict_block(
        df=df,
        assets=assets,
        features=features,
        top_trials=top_trials,
        train_idx=block2_train_idx,
        test_idx=block2_test_idx,
        dates_index=dates_index,
        seed=seed,
        logger=logger,
        spreads=spreads,
    )

    block2_bl = run_black_litterman_backtest(
        dates_target=dates_index[block2_test_idx + 1],
        y_true=block2["y_raw"][block2_test_idx],
        y_pred=block2["ensemble_pred_full"][block2_test_idx],
        mask=block2["test_pred_mask"],
        asset_names=assets,
        train_cov=block2["train_cov"],
        tau=tau,
        risk_aversion=risk_aversion,
        omega_scale=omega_scale,
        long_only=long_only,
        max_weight=max_weight,
    )

    final_test_weights = pd.concat(
        [block1_bl["weights_df"], block2_bl["weights_df"]],
        axis=0
    ).sort_index()

    final_test_returns = pd.concat(
        [block1_bl["returns_df"], block2_bl["returns_df"]],
        axis=0
    ).sort_index()

    final_test_metrics = portfolio_metrics_from_returns(
        final_test_returns["portfolio_return"].to_numpy()
    )

    if len(final_test_weights) > 1:
        turnover = np.abs(final_test_weights.diff().fillna(0.0)).sum(axis=1)
        final_test_metrics["avg_daily_turnover"] = float(turnover.mean())
    else:
        final_test_metrics["avg_daily_turnover"] = np.nan

    final_test_weights.to_csv(os.path.join(final_dir, "test_bl_weights.csv"))
    final_test_returns.to_csv(os.path.join(final_dir, "test_bl_returns.csv"))

    result = {
        "config": {
            "top_n": int(top_n),
            "seed": int(seed),
            "train_target_end": train_target_end,
            "test_year1_start": test_year1_start,
            "test_year1_end": test_year1_end,
            "test_year2_start": test_year2_start,
            "test_year2_end": test_year2_end,
            "mid_test_refit": True,
            "tau": _safe_float(tau),
            "risk_aversion": _safe_float(risk_aversion),
            "omega_scale": _safe_float(omega_scale),
            "long_only": bool(long_only),
            "max_weight": _safe_float(max_weight) if max_weight is not None else None,
        },
        "black_litterman": {
            "test_only": {
                "portfolio_metrics": {k: _safe_float(v) for k, v in final_test_metrics.items()},
                "returns": _records_from_df(final_test_returns, date_col_name="target_date"),
                "weights": _records_from_df(final_test_weights, date_col_name="target_date"),
            }
        },
    }

    with open(os.path.join(final_dir, "final_result.json"), "w") as f:
        json.dump(result, f, indent=2)

    print("\n=== FINAL TEST ONLY (MID-TEST REFIT) ===")
    print("Test BL metrics:", final_test_metrics)
    print("Saved weights:", os.path.join(final_dir, "test_bl_weights.csv"))
    print("Saved returns:", os.path.join(final_dir, "test_bl_returns.csv"))

    return result

## 18) Black–Litterman HPO

In [ ]:
# Separate Optuna study that tunes BL hyperparameters (tau, risk_aversion,
# view_confidence, lookback_days) on walk-forward validation Sharpe.
# Runs after the LSTM study so the model ensemble is already fixed.

def _build_topn_ensemble_predictions_for_split(
    df: pd.DataFrame,
    assets: List[str],
    features: Dict[str, List[str]],
    top_trials: List[optuna.trial.FrozenTrial],
    split_train_idx: np.ndarray,
    split_val_idx: np.ndarray,
    dates_index: pd.DatetimeIndex,
    seed: int,
    logger: logging.Logger,
    spreads: Optional[np.ndarray] = None,
) -> Dict[str, Any]:
    """
    Refit each top-N model on this split with ONE seed, predict val, then ensemble.
    Also return train covariance estimated only from split train raw targets.
    """
    T = len(df)
    N = len(assets)

    pred_full_list = []
    canonical_y_raw = None
    canonical_y_mask = None

    for rank, trial in enumerate(top_trials, start=1):
        picks = pick_features_from_best_params(trial.params, features)
        lookback = int(trial.params["lookback"])
        model_params = build_model_params_from_best_params(trial.params)

        _, X_raw, y_model, y_raw, y_mask, feat_names, market_raw = build_raw_arrays(
            df=df,
            assets=assets,
            picks=picks,
            features=features,
            logger=logger,
            spreads=spreads,
        )

        if canonical_y_raw is None:
            canonical_y_raw = y_raw.copy()
            canonical_y_mask = y_mask.copy()

        X_train_raw = X_raw[split_train_idx]
        market_train_raw = market_raw[split_train_idx] if market_raw is not None else None

        fitted, market_ft = fit_transforms_on_train(X_train_raw, feat_names, market_train_raw)
        X_trans, market_trans = apply_transforms(X_raw, fitted, market_raw, market_ft)

        sample_valid = compute_sample_validity(
            X_raw=X_trans,
            y_mask=y_mask,
            lookback=lookback,
            min_valid_targets=max(1, len(assets) // 3),
        )

        train_core_idx, es_idx = split_train_for_early_stop(
            train_idx=split_train_idx,
            dates_index=dates_index,
            es_years=1,
            min_train_days=252,
        )

        train_core_t = train_core_idx[sample_valid[train_core_idx]]
        es_t = es_idx[sample_valid[es_idx]]
        val_t = split_val_idx[sample_valid[split_val_idx]]

        if len(train_core_t) < 64 or len(es_t) < 32 or len(val_t) < 32:
            logger.warning(
                f"Skipping top trial {trial.number} on split because samples are too small: "
                f"train_core={len(train_core_t)} es={len(es_t)} val={len(val_t)}"
            )
            continue

        input_dim_per_asset = X_trans.shape[2] + (1 if market_trans is not None else 0)

        model, _ = train_one_model(
            X=X_trans,
            y=y_model,
            y_mask=y_mask,
            market=market_trans,
            train_t=train_core_t,
            es_t=es_t,
            lookback=lookback,
            input_dim_per_asset=input_dim_per_asset,
            output_dim=len(assets),
            params=model_params,
            seed=seed,
            logger=logger,
        )

        val_eval = evaluate_model_on_t_indices(
            model=model,
            X=X_trans,
            y=y_model,
            y_mask=y_mask,
            market=market_trans,
            t_idx=val_t,
            lookback=lookback,
            batch_size=model_params["batch_size"],
            loss_name=model_params["loss_name"],
            huber_delta=model_params["huber_delta"],
        )

        pred_full = np.full((T, N), np.nan, dtype=np.float32)
        if len(val_eval["t_idx"]) > 0:
            pred_full[val_eval["t_idx"]] = val_eval["y_pred"]

        pred_full_list.append(pred_full)

    if len(pred_full_list) == 0:
        raise ValueError("No usable top-N model predictions were produced for this split.")

    ensemble_pred_full = nanmean_predictions(pred_full_list)

    train_cov = estimate_train_covariance(
        y_train=canonical_y_raw[split_train_idx],
        y_mask_train=canonical_y_mask[split_train_idx],
        ridge=1e-5,
    )

    return {
        "ensemble_pred_full": ensemble_pred_full,
        "y_raw": canonical_y_raw,
        "y_mask": canonical_y_mask,
        "train_cov": train_cov,
    }


def objective_factory_bl(
    df: pd.DataFrame,
    assets: List[str],
    features: Dict[str, List[str]],
    model_study: optuna.Study,
    base_log_dir: str,
    seed: int = SEED_PORTFOLIO_LSTM_RETURNS_V5_2_2,
    spreads: Optional[np.ndarray] = None,
):
    dates_index = pd.DatetimeIndex(df.index)
    splits = make_walk_forward_splits(dates_index)
    # top_trials fetched inside objective so top_n can vary per trial

    def objective(trial: optuna.Trial) -> float:
        trial_dir = os.path.join(base_log_dir, f"bl_trial_{trial.number:05d}")
        #os.makedirs(trial_dir, exist_ok=True)
        logger = setup_logger(trial_dir)

        tau = trial.suggest_float("tau", 1e-3, 0.5, log=True)
        risk_aversion = trial.suggest_float("risk_aversion", 0.5, 10.0, log=True)
        omega_scale = trial.suggest_float("omega_scale", 0.01, 2.0, log=True)
        max_weight = trial.suggest_float("max_weight", 0.10, 1.0)
        long_only = bool(trial.suggest_categorical("long_only", [1]))
        top_n = trial.suggest_categorical("top_n", [3, 5, 10])
        top_trials = get_top_n_completed_trials(model_study, top_n=top_n)

        logger.info("=" * 100)
        logger.info(
            f"BL TRIAL {trial.number} | tau={tau:.6f} risk_aversion={risk_aversion:.6f} "
            f"omega_scale={omega_scale:.6f} max_weight={max_weight:.4f} "
            f"long_only={long_only} top_n={top_n}"
        )

        split_sharpes = []

        for split_i, (train_idx, val_idx) in enumerate(splits):
            logger.info("-" * 100)
            logger.info(
                f"[BL SPLIT {split_i}] "
                f"train {df.index[train_idx[0]]}..{df.index[train_idx[-1]]} | "
                f"val {df.index[val_idx[0]]}..{df.index[val_idx[-1]]}"
            )

            try:
                pred_bundle = _build_topn_ensemble_predictions_for_split(
                    df=df,
                    assets=assets,
                    features=features,
                    top_trials=top_trials,
                    split_train_idx=train_idx,
                    split_val_idx=val_idx,
                    dates_index=dates_index,
                    seed=seed,
                    logger=logger,
                    spreads=spreads,
                )
            except Exception as e:
                logger.info(f"[PRUNE] Failed to build ensemble predictions on split {split_i}: {e}")
                raise optuna.TrialPruned()

            ensemble_pred_full = pred_bundle["ensemble_pred_full"]
            y_raw = pred_bundle["y_raw"]
            y_mask = pred_bundle["y_mask"]
            train_cov = pred_bundle["train_cov"]

            val_pred_mask = y_mask[val_idx] & np.isfinite(ensemble_pred_full[val_idx])

            bl_val = run_black_litterman_backtest(
                dates_target=dates_index[val_idx + 1],
                y_true=y_raw[val_idx],
                y_pred=ensemble_pred_full[val_idx],
                mask=val_pred_mask,
                asset_names=assets,
                train_cov=train_cov,
                tau=tau,
                risk_aversion=risk_aversion,
                omega_scale=omega_scale,
                long_only=long_only,
                max_weight=max_weight,
            )

            split_sharpe = bl_val["metrics"]["sharpe"]
            if not np.isfinite(split_sharpe):
                split_sharpe = -999.0

            split_sharpes.append(float(split_sharpe))

            logger.info(
                f"[BL SPLIT {split_i}] sharpe={split_sharpe:.6f} "
                f"ann_return={bl_val['metrics']['ann_return']:.6f} "
                f"ann_vol={bl_val['metrics']['ann_vol']:.6f} "
                f"max_dd={bl_val['metrics']['max_drawdown']:.6f}"
            )

            trial.report(float(np.mean(split_sharpes)), step=split_i)
            if trial.should_prune():
                logger.info("[PRUNE] BL Optuna pruned this trial.")
                raise optuna.TrialPruned()

        final_score = float(np.mean(split_sharpes))
        trial.set_user_attr("mean_val_sharpe", final_score)

        logger.info("=" * 100)
        logger.info(f"BL TRIAL {trial.number} DONE | mean_val_sharpe={final_score:.6f}")

        return final_score

    return objective


def run_black_litterman_optuna_search(
    df: pd.DataFrame,
    assets: List[str],
    features: Dict[str, List[str]],
    model_study: optuna.Study,
    out_dir: str,
    seed: int = SEED_PORTFOLIO_LSTM_RETURNS_V5_2_2,
    n_trials: int = 30,
    spreads: Optional[np.ndarray] = None,
):
    bl_dir = os.path.join(out_dir, "bl_optuna_search")
    #os.makedirs(bl_dir, exist_ok=True)

    storage_path = os.path.join(out_dir, "black_litterman_optuna.db")
    storage = f"sqlite:///{storage_path}"

    study = optuna.create_study(
        study_name="black_litterman_hpo",
        direction="maximize",
        storage=storage,
        load_if_exists=True,
        sampler=TPESampler(seed=None, multivariate=True, group=True, constant_liar=True),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
    )

    objective = objective_factory_bl(
        df=df,
        assets=assets,
        features=features,
        model_study=model_study,
        base_log_dir=bl_dir,
        seed=seed,
        spreads=spreads,
    )

    study.optimize(objective, n_trials=n_trials, gc_after_trial=True, show_progress_bar=True)

    best = study.best_trial
    summary = {
        "best_value": best.value,
        "best_params": best.params,
        "best_user_attrs": best.user_attrs,
        "n_trials": len(study.trials),
        "single_seed_used": int(seed),
    }

    with open(os.path.join(out_dir, "best_bl_trial.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print("\n=== BEST BL TRIAL ===")
    print("Value (mean validation Sharpe):", best.value)
    print("Params:", best.params)
    print("DB:", storage_path)

    return study

## 19) Run

In [ ]:
# ── Load data ────────────────────────────────────────────────────────────
df = pd.read_parquet(DATASET_PATH)

with open(SPREADS_PATH, 'r') as f:
    spread_dict = json.load(f)
spreads = np.array([spread_dict[a] for a in ASSETS], dtype=np.float32)


# ── 1) LSTM hyperparameter search ─────────────────────────────────────────
study, out_dir = run_optuna_search(
    df=df, assets=ASSETS, features=FEATURES,
    out_dir=OUT_DIR, n_trials=N_TRIALS,
    seeds=MODEL_SEEDS, spreads=spreads,
)

suggested_lstm = estimate_search_space(study)
total_n_trials = len(study.trials)


# ── 2) Refit best trial on all walk-forward splits ────────────────────────
# summary_df = fit_best_trial_and_export(
#     df=df, assets=ASSETS, features=FEATURES,
#     study=study, out_dir=out_dir,
#     seeds=MODEL_SEEDS, spreads=spreads,
# )
# print('\n=== BEST REFIT SUMMARY ===')
# print(summary_df.groupby('split').mean(numeric_only=True))


# ── 3) Black-Litterman HPO on validation Sharpe ───────────────────────────
bl_study = run_black_litterman_optuna_search(
    df=df, assets=ASSETS, features=FEATURES,
    model_study=study, out_dir=out_dir,
    seed=BL_SEED,
    n_trials=N_TRIALS_BL, spreads=spreads,
)

bl_best_params = bl_study.best_trial.params
print('\n=== BEST BL PARAMS ===')
print(bl_best_params)

suggested_bl = estimate_search_space(bl_study)
total_n_trials_bl = len(bl_study.trials)


# ── 4) Final test with mid-test refit ─────────────────────────────────────
final_result = final_topn_black_litterman_eval_with_midtest_refit(
    df=df, assets=ASSETS, features=FEATURES,
    study=study, out_dir=out_dir,
    seed=BL_SEED,
    train_target_end='2023-06-30',
    test_year1_start='2023-07-01', test_year1_end='2024-06-30',
    test_year2_start='2024-07-01', test_year2_end='2025-06-30',
    bl_best_params=bl_best_params, spreads=spreads,
)


In [ ]:
print(f'LSTM: You used N_TRIALS={total_n_trials}. Suggested ceiling: {suggested_lstm} x continuous.')
print(f'Black-Litterman: You used N_TRIALS_BL={total_n_trials_bl}. Suggested ceiling: {suggested_bl} x continuous.')


## 20) Portfolio metrics & plots

In [ ]:
weights = np.array([[row[c] for c in ASSETS] for row in final_result['black_litterman']['test_only']['weights']])


In [ ]:
import sys
from pathlib import Path
from src.metrics import PortfolioMetrics

pm = PortfolioMetrics()
print(pm.summary(weights))
pm.plot_weights(weights)
pm.plot_cumulative_returns(weights)


In [ ]:
np.save("LSTMvia-returns_technical_weights.npy", weights)